# 19. 모델 단일 학습 (Pre-split Subsets 사용)

**파이프라인 순서**

1. 설정 확인 및 사전 분할 데이터 감지 (필수) 
2. 앙상블 학습 (단일 실행)  
3. 결과 요약 및 시각화 (PR-AUC, 혼동행렬)  
4. SHAP 중요도 분석  
5. 모델 저장  

> **주의**: 20번 노트북을 통해 데이터를 먼저 분할해두어야 합니다.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from src.train_core import (
    run_training, print_ensemble_summary, 
    plot_subset_prauc, plot_confusion_matrix
)
import config.train_config as cfg

print('✅ 환경 준비 완료')
print(f'SEED             : {cfg.SEED}')
print(f'SUBSET_DIR       : {cfg.SUBSET_DIR}')
print(f'VAL_TUNE_PATH    : {cfg.VAL_TUNE_PATH}')

## 1. 데이터셋 존재 확인
사전 분할된 데이터가 없으면 실행을 중단합니다.

In [ ]:
subset_path = Path(cfg.SUBSET_DIR)
subset_files = sorted(list(subset_path.glob("subset_*.parquet")))

if not subset_files:
    raise FileNotFoundError(f"❌ [ERROR] {cfg.SUBSET_DIR} 경로에 분할된 데이터가 없습니다.")

print(f"✅ 사전 분할 데이터 감지됨: {len(subset_files)}개 서브셋")

FEATURE_COLS = cfg.FEATURE_COLS
print(f"✅ 사용할 피처 수: {len(FEATURE_COLS) if FEATURE_COLS else '전체'}")

## 2. 훈련 및 검증 (단일 실행)
데이터 로딩과 학습 루프 내 검증 과정이 생략되어 매우 빠르게 진행됩니다.

In [ ]:
result_single = run_training(
    cfg=cfg,
    feature_cols=FEATURE_COLS,
    run_optuna=False,
    show_plots=True,
)

## 3. 결과 요약 및 시각화
앙상블 성능과 임계값별 혼동행렬(개수 및 비율)을 확인합니다.

In [ ]:
from sklearn.metrics import classification_report

ens_res = result_single['ensemble_result']

# 1) 텍스트 요약
print_ensemble_summary(ens_res)

# 2) 서브셋별 PR-AUC 바 차트
plot_subset_prauc(ens_res)

# 3) 혼동행렬 (개수 + 비율)
plot_confusion_matrix(ens_res, cfg)

# 4) 상세 분류 리포트
rpt_thr = cfg.REPORT_THRESHOLD
print(f'\n[Classification Report  threshold={rpt_thr}]')
print(classification_report(
    ens_res.val_tune_y_true, 
    (ens_res.val_tune_probs >= rpt_thr).astype(int), 
    target_names=['Normal', 'Failure']
))

## 4. SHAP 중요도 분석

In [ ]:
import shap
import warnings
warnings.filterwarnings('ignore')

ens_res      = result_single['ensemble_result']
feature_cols = result_single['feature_cols']
df_val_tune  = pd.read_parquet(cfg.VAL_TUNE_PATH) # SHAP용 샘플 추출을 위해 로드

X_val_shap = df_val_tune[feature_cols].copy()
SHAP_SAMPLE = min(1000, len(X_val_shap))
X_sample = X_val_shap.sample(SHAP_SAMPLE, random_state=cfg.SEED).reset_index(drop=True)

print(f'SHAP 계산 중... (샘플={SHAP_SAMPLE}, 모델={len(ens_res.models)}개)')

sv_list = []
for i, model in enumerate(ens_res.models):
    explainer = shap.TreeExplainer(model)
    sv = explainer.shap_values(X_sample)
    if isinstance(sv, list): sv = sv[1]
    sv_list.append(sv)
    print(f'  모델 {i+1}/{len(ens_res.models)} 완료', end='\r')

mean_shap = np.mean(sv_list, axis=0)
print('\nSHAP 계산 완료!')

shap.summary_plot(mean_shap, X_sample, feature_names=feature_cols, max_display=30, show=True)

## 5. 모델 저장

In [ ]:
import joblib, json
from pathlib import Path

SAVE_DIR = Path(cfg.MODEL_SAVE_DIR)
SAVE_DIR.mkdir(parents=True, exist_ok=True)

for i, model in enumerate(result_single['ensemble_result'].models):
    path = SAVE_DIR / f'subset_{i:02d}.pkl'
    joblib.dump(model, path)
    print(f'  저장: {path}')

with open(SAVE_DIR / 'feature_cols.json', 'w', encoding='utf-8') as f:
    json.dump(result_single['feature_cols'], f, ensure_ascii=False, indent=2)

with open(SAVE_DIR / 'used_params.json', 'w', encoding='utf-8') as f:
    json.dump(result_single['best_params'], f, ensure_ascii=False, indent=2)

print(f'\n✨ 모든 모델 및 설정 저장 완료: {SAVE_DIR}')